In [1]:
import os
from datetime import datetime

import torch
from torch import nn
from torchvision import datasets
import torchvision.transforms.v2 as transforms_v2
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torcheval.metrics.functional import (
    multiclass_accuracy,
    multiclass_f1_score
)
from torchvision.ops import generalized_box_iou

In [2]:
from roboflow import Roboflow
rf = Roboflow(api_key="D2P3bCoryjlMEaJoExwn")
project = rf.workspace("wrkspc-gi0hz").project("cloud-classification-mf91q")
version = project.version(7)
dataset = version.download("tensorflow")

loading Roboflow workspace...
loading Roboflow project...


In [3]:
CLASS_NAMES = [
    "altocumulus",
    "altostratus",
    "cirrocumulus",
    "cirrostratus",
    "cirrus",
    "cumulonimbus",
    "cumulus",
    "nimbostratus",
    "stratocumulus",
    "stratus",
]

In [4]:
IMG_SIZE = (224, 224)

from torchvision.io import read_image

def prepare_dataset(records, image_path):
    images = []
    targets = []
    labels = []
    for index, row in records.iterrows():
        (filename, width, height, class_name, xmin, ymin, xmax, ymax) = row
        
        fullpath = os.path.join(image_path, filename)
        img = read_image(fullpath)
        
        # Convert into porportion of the image size
        xmin = round(xmin/ width, 2)
        ymin = round(ymin/ height, 2)
        xmax = round(xmax/ width, 2)
        ymax = round(ymax/ height, 2)
        
        images.append(img)
        targets.append((xmin, ymin, xmax, ymax))
        labels.append(CLASS_NAMES.index(class_name))
    return images, targets, labels

In [5]:
import os
import pandas as pd

# Train set
TRAINING_CSV_FILE = 'Cloud-Classification-7/train/_annotations.csv'
TRAINING_IMAGE_DIR = 'Cloud-Classification-7/train'

training_image_records = pd.read_csv(TRAINING_CSV_FILE)

train_image_path = os.path.join(os.getcwd(), TRAINING_IMAGE_DIR)

train_images, train_targets, train_labels = prepare_dataset(training_image_records, train_image_path)

# Validate set
VALIDATING_CSV_FILE = 'Cloud-Classification-7/valid/_annotations.csv'
VALIDATING_IMAGE_DIR = 'Cloud-Classification-7/valid'

validating_image_records = pd.read_csv(VALIDATING_CSV_FILE)

valid_image_path = os.path.join(os.getcwd(), VALIDATING_IMAGE_DIR)

valid_images, valid_targets, valid_labels = prepare_dataset(validating_image_records, valid_image_path)

# Testing set
TESTING_CSV_FILE = 'Cloud-Classification-7/test/_annotations.csv'
TESTING_IMAGE_DIR = 'Cloud-Classification-7/test'

testing_image_records = pd.read_csv(TESTING_CSV_FILE)

test_image_path = os.path.join(os.getcwd(), TESTING_IMAGE_DIR)

test_images, test_targets, test_labels = prepare_dataset(testing_image_records, test_image_path)

In [6]:
train_images = torch.stack(train_images).float() / 255.0
train_targets = torch.tensor(train_targets, dtype=torch.float32)
train_labels = torch.tensor(train_labels, dtype=torch.long)

valid_images = torch.stack(valid_images).float() / 255.0
valid_targets = torch.tensor(valid_targets, dtype=torch.float32)
valid_labels = torch.tensor(valid_labels, dtype=torch.long)

test_images = torch.stack(test_images).float() / 255.0
test_targets = torch.tensor(test_targets, dtype=torch.float32)
test_labels = torch.tensor(test_labels, dtype=torch.long)

In [7]:
num_classes = len(CLASS_NAMES)

#create the common input layer
input_shape = (IMG_SIZE[0], IMG_SIZE[1], 3)

In [8]:
def bbox_loss(pred_boxes, target_boxes):
    giou = generalized_box_iou(pred_boxes, target_boxes)
    return 1 - giou.mean()

In [10]:
from torch import nn

losses = {
    "cl_head": nn.CrossEntropyLoss(),  # Loss for class prediction
    "bb_head": bbox_loss # Loss for bounding box prediction
}

In [11]:
trainTargets = {
    "cl_head": train_labels,
    "bb_head": train_targets
}
validTargets = {
    "cl_head": valid_labels,
    "bb_head": valid_targets
}
training_epochs = 20

In [12]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights


if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

class MobileNetMultiHead(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        weights = MobileNet_V2_Weights.DEFAULT
        model = mobilenet_v2(weights=weights)

        # backbone (remove classifier)
        self.features = model.features

        # global average pooling
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        in_features = model.classifier[1].in_features
        
        # classification head
        self.cl_head = nn.Linear(in_features, num_classes)

        # bounding box head
        self.bb_head = nn.Linear(in_features, 4)

    def forward(self, x):

        # equivalent to preprocess_input must be done in transforms

        x = self.features(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)

        class_output = torch.softmax(self.cl_head(x), dim=1)
        bbox_output = torch.sigmoid(self.bb_head(x))

        return class_output, bbox_output

Using device: mps


In [13]:
model = MobileNetMultiHead(num_classes=len(CLASS_NAMES))
model = model.to(device)


fine_tune_at = 3
ct = 0
for child in model.children():
    ct += 1
    if ct < fine_tune_at:
        for param in child.parameters():
            param.requires_grad = False
        print(f"{child} ({ct}): {False}")
    else:
        for param in child.parameters():
            param.requires_grad = True
        print(f"{child} ({ct}): {True}")

Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU6(inplace=True)
  )
  (1): InvertedResidual(
    (conv): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (2): InvertedResidual(
    (conv): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (

In [14]:
def train_one_epoch(
        dataloader, model, losses, optimizer, 
        epoch, device, writer, log_step_interval=50
    ):
    size = len(dataloader.dataset)
    model.train()
    running_loss = 0

    for images, labels, boxes in dataloader:

        images = images.to(device)
        labels = labels.to(device)
        boxes = boxes.to(device)

        optimizer.zero_grad()

        cl_pred, bb_pred = model(images)
        
        loss = (
            losses["cl_head"](cl_pred, labels) +
            losses["bb_head"](bb_pred, boxes)
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if (i+1) % log_step_interval == 0:
            last_loss = running_loss / log_step_interval
            current = (i + 1) * len(images)
            print(f"loss: {last_loss:>7f}  [{current:>5d}/{size:>5d}]")
            # Log the running loss
            writer.add_scalar(
                'Loss/train_running',
                running_loss / 1000,
                epoch * len(dataloader) + i
            )
            running_loss = 0.



def test(dataloader, model, losses, device):
    # ----- validation -----
    model.eval()
    val_running_loss = 0
    y_preds = []
    y_trues = []

    with torch.no_grad():
        for images, labels, boxes in dataloader:

            images = images.to(device)
            labels = labels.to(device)
            boxes = boxes.to(device)

            cl_pred, bb_pred = model(images)
            
            loss = (
                losses["cl_head"](cl_pred, labels) +
                losses["bb_head"](bb_pred, boxes)
            )

            val_running_loss += loss.item()

            y_preds.append(cl_pred)
            y_trues.append(labels)

    y_preds = torch.cat(y_preds)
    y_trues = torch.cat(y_trues)

    val_loss = val_running_loss / len(dataloader)
    return val_loss, y_preds, y_trues

In [27]:
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter

learning_rate = 1e-3
batch_size = 64       
epochs = 5             
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
writer = SummaryWriter(f'./runs/trainer_{model._get_name()}_{datetime.now().strftime("%Y%m%d-%H%M%S")}')

In [28]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(train_images, train_labels, train_targets)
valid_ds = TensorDataset(valid_images, valid_labels, valid_targets)
test_ds = TensorDataset(test_images, test_labels, test_targets)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size=32, shuffle=False)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=False)

In [29]:
batch = next(iter(train_dl))

print(type(batch))
print(len(batch))

for i, item in enumerate(batch):
    print(i, type(item))

<class 'list'>
3
0 <class 'torch.Tensor'>
1 <class 'torch.Tensor'>
2 <class 'torch.Tensor'>


In [30]:
from torcheval.metrics.functional import (
    multiclass_accuracy,
    multiclass_f1_score
)

best_vloss = 100000.
for epoch in range(epochs):
    print(f"Epoch {epoch+1} / {epochs}")
    train_one_epoch(train_dl, model, losses, optimizer, epoch, device, writer, log_step_interval=1)
    train_loss, train_y_preds, train_y_trues = test(train_dl, model, losses, device)
    val_loss, val_y_preds, val_y_trues = test(valid_dl, model, losses, device)
    
    # Performance metrics
    train_perf = {
        'accuracy': multiclass_accuracy(train_y_preds, train_y_trues).item(),
        'f1': multiclass_f1_score(train_y_preds, train_y_trues).item(),
    }
    
    # Performance metrics
    val_perf = {
        'accuracy': multiclass_accuracy(val_y_preds, val_y_trues).item(),
        'f1': multiclass_f1_score(val_y_preds, val_y_trues).item(),
    }
    
    # Log model training performance
    writer.add_scalars('Train vs. Valid/loss', 
        {'train':train_loss, 'valid': val_loss}, 
        epoch)
    writer.add_scalars(
        'Performance/acc', 
        {'train':train_perf['accuracy'], 'valid': val_perf['accuracy']},
        epoch)
    writer.add_scalars(
        'Performance/f1', 
        {'train':train_perf['f1'], 'valid': val_perf['f1']},
        epoch)

    # Track best performance, and save the model's state
    if val_loss < best_vloss:
        best_vloss = val_loss
        torch.save(model.state_dict(), 'model_best_vloss.pth')
        print('Saved best model to model_best_vloss.pth')
print("Done!")

Epoch 1 / 5
loss: 1.906082  [   96/ 1558]
loss: 0.663881  [   96/ 1558]
loss: 1.541626  [   96/ 1558]
loss: 0.800791  [   96/ 1558]
loss: 1.012179  [   96/ 1558]
loss: 0.679897  [   96/ 1558]
loss: 1.619202  [   96/ 1558]
loss: 1.870590  [   96/ 1558]
loss: -3.763382  [   96/ 1558]
loss: 0.925418  [   96/ 1558]
loss: 1.264019  [   96/ 1558]
loss: 1.469092  [   96/ 1558]
loss: 1.687335  [   96/ 1558]
loss: 1.555141  [   96/ 1558]
loss: 1.467817  [   96/ 1558]
loss: 0.805402  [   96/ 1558]
loss: 1.663944  [   96/ 1558]
loss: 1.390770  [   96/ 1558]
loss: -0.795404  [   96/ 1558]
loss: 1.850985  [   96/ 1558]
loss: -2.291671  [   96/ 1558]
loss: -0.308908  [   96/ 1558]
loss: 1.434851  [   96/ 1558]
loss: 0.581505  [   96/ 1558]
loss: -1.155901  [   96/ 1558]
loss: 1.449861  [   96/ 1558]
loss: 1.576052  [   96/ 1558]
loss: 1.147276  [   96/ 1558]
loss: 1.265797  [   96/ 1558]
loss: 1.319825  [   96/ 1558]
loss: 1.753797  [   96/ 1558]
loss: -0.658821  [   96/ 1558]
loss: 1.375204  [   96

In [31]:
model_best = MobileNetMultiHead(num_classes=len(CLASS_NAMES))
model_best = model_best.to(device)
model_best.load_state_dict(torch.load("model_best_vloss.pth"))

train_loss, train_y_preds, train_y_trues = test(train_dl, model_best, losses, device)

# Performance metrics on the training set
train_perf = {
    'accuracy': multiclass_accuracy(train_y_preds, train_y_trues).item(),
    'f1': multiclass_f1_score(train_y_preds, train_y_trues).item(),
}

# TODO: Use the best model on the test set
test_loss, test_y_preds, test_y_trues = test(test_dl, model_best, losses, device)

# Performance metrics
test_perf = {
    'accuracy': multiclass_accuracy(test_y_preds, test_y_trues).item(),
    'f1': multiclass_f1_score(test_y_preds, test_y_trues).item(),
}

print(f"Train: loss={train_loss:>8f}, acc={(100*train_perf['accuracy']):>0.1f}%, f1={(100*train_perf['f1']):>0.1f}%")
print(f"Test: loss={test_loss:>8f}, acc={(100*test_perf['accuracy']):>0.1f}%, f1={(100*test_perf['f1']):>0.1f}%")

Train: loss=0.629468, acc=61.2%, f1=61.2%
Test: loss=1.265004, acc=58.1%, f1=58.1%
